# Global Orchestrator — user perspective

**What it does.** Pure-Python class. Reads Linear state, applies the todo + dependency filter, sorts via the Risk Agent priority queue, and dispatches UAT for User Stories whose child tasks are all QA-approved.

**No LLM, no SDK session.** Just deterministic Python over Linear data fetched via `run_validated_linear_agent`.

**Entry point.** `await GlobalOrchestrator().get_ready_tasks() -> GlobalOrchestratorOutput`.

**Phase 5 surface.** Risk-sorted priority queue (D-10), UAT inline-await dispatch (UATA-01 / D-01), G6 cycle-cap escalation, G10 pre-persist validation, RISK-04 / D-09 (improvement_triggers always `[]` in the per-cycle path).

In [ ]:
import sys
from pathlib import Path

# Robust _helpers.py discovery — works whether jupyter lab was launched from
# the repo root, notebooks/, or notebooks/agents/. The loop walks up from cwd
# until it finds the parent that holds _helpers.py and inserts it on sys.path.
nb_dir = Path.cwd()
for _parent in (nb_dir, *nb_dir.parents):
    if (_parent / "_helpers.py").exists():
        sys.path.insert(0, str(_parent))
        break

from _helpers import ensure_src_on_path, runtime_summary

ROOT = ensure_src_on_path()
print("HSB_RUNTIME_<AGENT> selection:\n" + runtime_summary())

## Pure-logic probe — `_filter_ready_items`

The dependency filter is independent of Linear and exercises GORD-01/02. Stub a Linear-shaped state and watch which tasks survive.

In [ ]:
from hsb.agents.global_orchestrator import GlobalOrchestrator

go = GlobalOrchestrator()
items = [
    {"id": "LIN-A", "status": "done", "dependencies": []},
    {"id": "LIN-B", "status": "todo", "dependencies": []},
    {"id": "LIN-C", "status": "todo", "dependencies": ["LIN-A"]},  # ready (dep done)
    {"id": "LIN-D", "status": "todo", "dependencies": ["LIN-B"]},  # blocked (dep todo)
]
ready = go._filter_ready_items(items)
print("ready ids:", [r["id"] for r in ready])
assert {r["id"] for r in ready} == {"LIN-B", "LIN-C"}, "filter regression"
print("GORD-01/02 holding")

## Live invocation

Gated on `HSB_NOTEBOOK_RUN_LIVE=1`. Reads Linear state for the current project. The result includes the priority-sorted ready tasks, EPIC readiness signal, and UAT dispatched IDs.

In [ ]:
from _helpers import assert_g1_safe, gated, live_mode

if not live_mode():
    print(gated("GlobalOrchestrator.get_ready_tasks live run"))
else:
    assert_g1_safe()

    # Top-level await — IPython already runs the cell inside an active event
    # loop, so asyncio.run() would refuse to nest.
    out = await go.get_ready_tasks()  # noqa: F704
    print("ready:    ", [t.id for t in out.ready_tasks])
    print("backlog empty:", out.is_backlog_empty)
    print("epic ready:   ", out.is_epic_ready)
    print("UAT dispatched:", out.uat_dispatched)